# Experiment C (HyperAttDDI): Full Training & Post-Training Adverse Event Screening Engine

## Summary of Architecture & Clinical Workflow
1. **Architecture (Exp C):**
   - **Drug Encoder:** ChemBERTa (768d) $\to$ Linear(768 $\to$ 256) $\to$ ReLU $\to$ Dropout
   - **Hypergraph Message Passing:** 2-layer PyG `HypergraphConv(256, 64, heads=4, use_attention=True)`
   - **Side-Effect Encoder:** SapBERT (768d) $\to$ Linear(768 $\to$ 256) $\to$ ReLU $\to$ Dropout
   - **SE-Conditioned Attention Pooling:** Query dynamically conditioned on both combination state and adverse event:
     $$q = W_q([h_{e, \text{mean}} \parallel h_{se}])$$
   - **Interaction Decoder:** $[h_{combo} \parallel h_{se}] \to \text{MLP} \to P(y=1)$

2. **Training Phase (High-Quality Ground Truth):**
   - Trained on statistically validated FDA FAERS adverse event reports ($>90\%$ confidence).

3. **Post-Training Full-Spectrum Screening Engine:**
   - Takes any drug combination (2 to 8 drugs) and screens against **all 7,350 MedDRA adverse events in parallel** (<50 ms).
   - Applies a **High-Safety Medical Alert Threshold ($30\%$)** to ensure near-zero missed adverse reactions (High Sensitivity/Recall).
   - Categorizes findings into **🔴 High Risk ($\ge 70\%$)**, **🟡 Moderate Risk ($30\%-70\%$)**, and **🟢 Low Risk ($< 30\%$)** with culprit drug attribution.

In [ ]:
# ------------------------------------------------------------------------------
# Step 1: Install PyTorch Geometric in Kaggle
# ------------------------------------------------------------------------------
!pip install -q torch_geometric

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name   : {torch.cuda.get_device_name(0)}")

In [ ]:
# ------------------------------------------------------------------------------
# Step 2: Core Imports & HyperAttDDI Model Definition
# ------------------------------------------------------------------------------
import os
import ast
import math
import random
import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch_geometric.nn as hnn
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

class HyperAttDDI_SE(nn.Module):
    """
    HyperAttDDI with Side-Effect Conditioned Attention (Exp C).
    """
    def __init__(self, in_dim=768, emb_dim=256, conv_dim=64, heads=4, d=64, p=0.1):
        super().__init__()
        self.d = d
        self.p = p
        
        # Module 1: Drug Encoder (ChemBERTa projection)
        self.drug_encoder = nn.Sequential(
            nn.Linear(in_dim, emb_dim),
            nn.ReLU(),
            nn.Dropout(p=p)
        )
        
        # Module 2: Hypergraph Multi-Head Attention Convolutions
        self.conv1 = hnn.HypergraphConv(emb_dim, conv_dim, heads=heads, use_attention=True, dropout=p)
        self.conv2 = hnn.HypergraphConv(conv_dim * heads, conv_dim, heads=heads, use_attention=True, dropout=p)
        
        # Module 4: Side-Effect Encoder (SapBERT projection)
        self.se_encoder = nn.Sequential(
            nn.Linear(in_dim, emb_dim),
            nn.ReLU(),
            nn.Dropout(p=p)
        )
        
        # Module 3: SE-Conditioned Attention Pooling
        self.W_q = nn.Linear(emb_dim * 2, d)  # Query carries both combination mean + side effect
        self.W_k = nn.Linear(emb_dim, d)
        self.W_v = nn.Linear(emb_dim, emb_dim)
        
        # Module 5: Interaction Decoder
        self.decoder = nn.Sequential(
            nn.Linear(emb_dim * 2, 128),  # [h_combo || h_se]
            nn.ReLU(),
            nn.Dropout(p=p),
            nn.Linear(128, 1)
        )
        
    def forward(self, drug_features, inc_matrix, se_features, return_attention=False):
        N, E = inc_matrix.shape
        H_T = inc_matrix.T  # (E, N)
        
        # 1. Drug Encoder
        X = self.drug_encoder(drug_features)  # (N, 256)
        
        # 2. Hypergraph Message Passing
        row, col = torch.where(H_T)
        edges = torch.cat([col.view(1, -1), row.view(1, -1)], dim=0).to(drug_features.device)
        degree_e = H_T.sum(dim=1, keepdim=True).clamp(min=1)
        
        attr1 = (H_T @ X) / degree_e
        X1 = F.relu(self.conv1(X, edges, hyperedge_attr=attr1))
        X1 = F.dropout(X1, p=self.p, training=self.training)
        
        attr2 = (H_T @ X1) / degree_e
        X2 = F.relu(self.conv2(X1, edges, hyperedge_attr=attr2))
        X2 = F.dropout(X2, p=self.p, training=self.training)
        
        # 4. Side Effect Encoder
        h_se = self.se_encoder(se_features)  # (E, 256)
        
        # 3. SE-Conditioned Attention Pooling
        h_e_mean = (H_T @ X2) / degree_e  # (E, 256)
        q_input = torch.cat([h_e_mean, h_se], dim=1)  # (E, 512)
        q = self.W_q(q_input)            # (E, d)
        k = self.W_k(X2)                  # (N, d)
        v = self.W_v(X2)                  # (N, 256)
        
        scores = torch.matmul(q, k.T) / math.sqrt(self.d)  # (E, N)
        scores = scores.masked_fill(H_T == 0, -1e9)
        alpha = F.softmax(scores, dim=1)  # (E, N)
        h_combo = torch.matmul(alpha, v)  # (E, 256)
        
        # 5. Decoder
        z = torch.cat([h_combo, h_se], dim=1)  # (E, 512)
        logits = self.decoder(z).squeeze(-1)    # (E,)
        
        if return_attention:
            return logits, alpha
        return logits

In [ ]:
# ------------------------------------------------------------------------------
# Step 3: Dataset Ingestion & Quarterly Cohort Loader
# ------------------------------------------------------------------------------
def find_dataset_path():
    if os.path.exists('/kaggle/input'):
        for root, dirs, files in os.walk('/kaggle/input'):
            if 'drug_embeddings_768d.pt' in files:
                return root
    candidates = ['data', '../data', '../../data', 'dataset', '../dataset', '../../dataset', '.']
    for c in candidates:
        if os.path.exists(c) and os.path.exists(os.path.join(c, 'drug_embeddings_768d.pt')):
            return c
    return '.'

def load_data(base_path, device):
    print(f"Searching dataset files in: {base_path}")
    
    subset_dir = None
    for cand in [
        os.path.join(base_path, 'evaluation_subset', 'subset_drug2-8_SE5-50'),
        os.path.join(base_path, 'subset_drug2-8_SE5-50'),
        os.path.join(base_path, 'dataset', 'evaluation_subset', 'subset_drug2-8_SE5-50'),
    ]:
        if os.path.exists(cand):
            subset_dir = cand
            break
    if subset_dir is None:
        for root, dirs, files in os.walk(base_path):
            if any('positive_samples' in f for f in files):
                subset_dir = root
                break

    print(f"Cohort CSV directory: {subset_dir}")
    
    training_sub_ds_names = ['2015Q1', '2015Q2', '2015Q3', '2016Q4', '2017Q1', '2017Q2', '2017Q3', '2017Q4', '2018Q3', '2019Q1', '2019Q2', '2019Q3', '2019Q4', '2020Q1', '2020Q2', '2020Q3', '2020Q4', '2021Q1', '2021Q3', '2021Q4', '2022Q1', '2022Q2', '2022Q3', '2022Q4', '2023Q1', '2023Q2', '2023Q3', '2023Q4', '2024Q2']
    validating_sub_ds_names = ['2014Q3', '2015Q4', '2016Q1', '2016Q3', '2021Q2', '2024Q1']
    testing_sub_ds_names = ['2014Q4', '2016Q2', '2018Q1', '2018Q2', '2018Q4', '2024Q3']
    
    # 1. Drug Embeddings
    emb_candidates = [
        os.path.join(base_path, "drug_embeddings_768d.pt"),
        os.path.join(base_path, "dataset", "drug_embeddings_768d.pt"),
    ]
    emb_path = None
    for cand in emb_candidates:
        if os.path.exists(cand):
            emb_path = cand
            break
    if emb_path is None:
        for root, dirs, files in os.walk(base_path):
            if 'drug_embeddings_768d.pt' in files:
                emb_path = os.path.join(root, 'drug_embeddings_768d.pt')
                break
                
    print(f"Loading drug embeddings: {emb_path}")
    drug_embeddings_dict = torch.load(emb_path, map_location='cpu', weights_only=False)
    
    # 2. Side Effect SapBERT Embeddings (7,350 CUIs)
    se_file = None
    for cand in [os.path.join(base_path, 'dictionary', 'Side_effects_unique.csv'), os.path.join(base_path, 'Side_effects_unique.csv')]:
        if os.path.exists(cand):
            se_file = cand
            break
    if se_file is None:
        for root, dirs, files in os.walk(base_path):
            if 'Side_effects_unique.csv' in files:
                se_file = os.path.join(root, 'Side_effects_unique.csv')
                break
        
    print(f"Loading Side-Effects dictionary: {se_file}")
    se_df = pd.read_csv(se_file)
    feature_cols = [str(i) for i in range(768)]
    se_embeddings_dict = {}
    se_names_dict = {}
    for _, row in se_df.iterrows():
        cui = row['umls_cui_from_meddra']
        se_embeddings_dict[cui] = row[feature_cols].values.astype(np.float32)
        se_names_dict[cui] = str(row['side_effect_name'])
        
    # 3. Drug Name Dictionary
    drug_names_dict = {}
    for root, dirs, files in os.walk(base_path):
        for f in files:
            if 'Drugbank_ID_SMILE' in f:
                d_df = pd.read_csv(os.path.join(root, f))
                drug_names_dict = dict(zip(d_df['DrugBank ID'], d_df['Name']))
                break
            
    def merge_sub_datasets(sub_datasets):
        pos_merged = pd.DataFrame()
        neg_merged = pd.DataFrame()
        for sub_ds in sub_datasets:
            pos_file = os.path.join(subset_dir, f'{sub_ds}_positive_samples_condition123_SE_above_0.9.csv')
            neg_file = os.path.join(subset_dir, f'{sub_ds}_negative_samples_condition123_SE_above_0.9.csv')
            if os.path.exists(pos_file):
                pos_merged = pd.concat([pos_merged, pd.read_csv(pos_file)], axis=0)
            if os.path.exists(neg_file):
                neg_merged = pd.concat([neg_merged, pd.read_csv(neg_file)], axis=0)
        return pos_merged, neg_merged

    print("Loading and merging quarterly cohorts...")
    train_data_pos, train_data_neg = merge_sub_datasets(training_sub_ds_names)
    val_data_pos, val_data_neg = merge_sub_datasets(validating_sub_ds_names)
    test_data_pos, test_data_neg = merge_sub_datasets(testing_sub_ds_names)
    
    all_drugs = set()
    for df in [train_data_pos, train_data_neg, val_data_pos, val_data_neg, test_data_pos, test_data_neg]:
        for drug_list in df['DrugBankID']:
            all_drugs.update([d for d in ast.literal_eval(drug_list) if d.lower() != 'none'])
            
    valid_drugs = sorted(list(d for d in all_drugs if d in drug_embeddings_dict))
    drug_to_index = {drug: idx for idx, drug in enumerate(valid_drugs)}
    num_drugs = len(valid_drugs)
    print(f"Total unique valid drugs in vocabulary: {num_drugs}")

    def build_incidence_and_se(pos_df, neg_df):
        num_pos = len(pos_df)
        num_neg = len(neg_df)
        
        inc_pos = np.zeros((num_drugs, num_pos), dtype=np.float32)
        se_pos = np.zeros((num_pos, 768), dtype=np.float32)
        pos_cuis = pos_df['SE_above_0.9'].values
        pos_drug_lists = pos_df['DrugBankID'].values
        
        for col_idx in range(num_pos):
            cui = pos_cuis[col_idx]
            if cui in se_embeddings_dict:
                se_pos[col_idx] = se_embeddings_dict[cui]
            for drug_id in ast.literal_eval(pos_drug_lists[col_idx]):
                if drug_id in drug_to_index:
                    inc_pos[drug_to_index[drug_id], col_idx] = 1.0
                    
        inc_neg = np.zeros((num_drugs, num_neg), dtype=np.float32)
        se_neg = np.zeros((num_neg, 768), dtype=np.float32)
        neg_cuis = neg_df['SE_above_0.9'].values
        neg_drug_lists = neg_df['DrugBankID'].values
        
        for col_idx in range(num_neg):
            cui = neg_cuis[col_idx]
            if cui in se_embeddings_dict:
                se_neg[col_idx] = se_embeddings_dict[cui]
            for drug_id in ast.literal_eval(neg_drug_lists[col_idx]):
                if drug_id in drug_to_index:
                    inc_neg[drug_to_index[drug_id], col_idx] = 1.0
                    
        labels_pos = np.ones(num_pos, dtype=np.float32)
        labels_neg = np.zeros(num_neg, dtype=np.float32)
        
        return inc_pos, inc_neg, se_pos, se_neg, labels_pos, labels_neg

    print("Constructing incidence hypergraph matrices...")
    train_inc_pos, train_inc_neg, train_se_pos, train_se_neg, train_lab_pos, train_lab_neg = build_incidence_and_se(train_data_pos, train_data_neg)
    val_inc_pos, val_inc_neg, val_se_pos, val_se_neg, val_lab_pos, val_lab_neg = build_incidence_and_se(val_data_pos, val_data_neg)
    test_inc_pos, test_inc_neg, test_se_pos, test_se_neg, test_lab_pos, test_lab_neg = build_incidence_and_se(test_data_pos, test_data_neg)
    
    extra_feature = torch.zeros((num_drugs, 768), dtype=torch.float32)
    for drug, idx in drug_to_index.items():
        extra_feature[idx] = drug_embeddings_dict[drug]
        
    train_inc = torch.tensor(np.concatenate([train_inc_pos, train_inc_neg], axis=1), dtype=torch.float32)
    train_se = torch.tensor(np.concatenate([train_se_pos, train_se_neg], axis=0), dtype=torch.float32)
    train_lab = torch.tensor(np.concatenate([train_lab_pos, train_lab_neg]), dtype=torch.float32)
    
    val_inc = torch.tensor(np.concatenate([val_inc_pos, val_inc_neg], axis=1), dtype=torch.float32)
    val_se = torch.tensor(np.concatenate([val_se_pos, val_se_neg], axis=0), dtype=torch.float32)
    val_lab = np.concatenate([val_lab_pos, val_lab_neg])
    
    test_inc = torch.tensor(np.concatenate([test_inc_pos, test_inc_neg], axis=1), dtype=torch.float32)
    test_se = torch.tensor(np.concatenate([test_se_pos, test_se_neg], axis=0), dtype=torch.float32)
    test_lab = np.concatenate([test_lab_pos, test_lab_neg])
    
    return {
        'train_inc': train_inc,
        'train_se': train_se,
        'train_lab': train_lab,
        'val_inc': val_inc,
        'val_se': val_se,
        'val_lab': val_lab,
        'test_inc': test_inc,
        'test_se': test_se,
        'test_lab': test_lab,
        'drug_features': extra_feature.to(device),
        'num_drugs': num_drugs,
        'drug_to_index': drug_to_index,
        'valid_drugs': valid_drugs,
        'se_df': se_df,
        'se_embeddings_dict': se_embeddings_dict,
        'se_names_dict': se_names_dict,
        'drug_names_dict': drug_names_dict,
        'test_pos_df': test_data_pos
    }

In [ ]:
# ------------------------------------------------------------------------------
# Step 4: Metric Evaluation & Attention Shift Diagnostics
# ------------------------------------------------------------------------------
def evaluate_dataset(model, drug_features, inc_matrix, se_matrix, labels, device, batch_size=512, threshold=0.50):
    model.eval()
    all_scores = []
    num_edges = inc_matrix.shape[1]
    
    with torch.no_grad():
        for i in range(0, num_edges, batch_size):
            batch_inc = inc_matrix[:, i:i+batch_size].to(device)
            batch_se = se_matrix[i:i+batch_size].to(device)
            logits = model(drug_features, batch_inc, batch_se)
            probs = torch.sigmoid(logits).cpu().numpy()
            all_scores.append(probs)
            
    all_scores = np.concatenate(all_scores, axis=0)
    binary_preds = (all_scores >= threshold).astype(int)
    
    return {
        'precision': precision_score(labels, binary_preds, zero_division=0),
        'recall': recall_score(labels, binary_preds, zero_division=0),
        'f1': f1_score(labels, binary_preds, zero_division=0),
        'auc': roc_auc_score(labels, all_scores),
        'prauc': average_precision_score(labels, all_scores),
        'scores': all_scores
    }

def perform_attention_visualization(model, drug_features, data, device, output_dir='.'):
    model.eval()
    drug_to_index = data['drug_to_index']
    se_embeddings_dict = data['se_embeddings_dict']
    drug_names_dict = data['drug_names_dict']
    num_drugs = data['num_drugs']
    
    target_ses = [
        ('C0741553', 'Hemorrhage / Bleeding'),
        ('C0151746', 'Renal Failure / Kidney Injury'),
        ('C0020649', 'Hypotension'),
        ('C0018790', 'Cardiac Arrest')
    ]
    
    sample_combos = [
        {
            'name': 'Cardiovascular Combo (Antiplatelet + Anticoagulant)',
            'drug_ids': ['DB00758', 'DB14726', 'DB00945'], # Clopidogrel, Dabigatran, Aspirin
        },
        {
            'name': 'CNS Neuro-Psychiatric Combo (SSRI + Benzo + Anticonvulsant + Opioid)',
            'drug_ids': ['DB01175', 'DB00349', 'DB00230', 'DB00193'], # Escitalopram, Clobazam, Pregabalin, Tramadol
        }
    ]
    
    print("\n=======================================================")
    print("      ATTENTION SHIFT DIAGNOSTICS TABLE (Exp C)        ")
    print("=======================================================")
    
    viz_rows = []
    with torch.no_grad():
        for combo in sample_combos:
            combo_name = combo['name']
            drug_ids = [d for d in combo['drug_ids'] if d in drug_to_index]
            drug_names = [drug_names_dict.get(d, d) for d in drug_ids]
            
            print(f"\n--- Drug Combination: {combo_name} ---")
            print(f"Drugs involved: {', '.join(drug_names)}")
            
            inc = np.zeros((num_drugs, len(target_ses)), dtype=np.float32)
            se_feats = np.zeros((len(target_ses), 768), dtype=np.float32)
            
            for s_idx, (cui, _) in enumerate(target_ses):
                for d in drug_ids:
                    inc[drug_to_index[d], s_idx] = 1.0
                if cui in se_embeddings_dict:
                    se_feats[s_idx] = se_embeddings_dict[cui]
                    
            inc_tensor = torch.tensor(inc, dtype=torch.float32).to(device)
            se_tensor = torch.tensor(se_feats, dtype=torch.float32).to(device)
            
            logits, alpha = model(drug_features, inc_tensor, se_tensor, return_attention=True)
            probs = torch.sigmoid(logits).cpu().numpy()
            alpha_np = alpha.cpu().numpy()
            
            print(f"{'Side Effect':<35} | " + " | ".join([f"{d[:12]:<12}" for d in drug_names]) + " | Predicted Prob")
            print("-" * (37 + 15 * len(drug_names) + 17))
            
            for s_idx, (cui, se_label) in enumerate(target_ses):
                weights = [alpha_np[s_idx, drug_to_index[d]] for d in drug_ids]
                prob = probs[s_idx]
                w_str = " | ".join([f"{w:12.4f}" for w in weights])
                print(f"{se_label:<35} | {w_str} | {prob:14.4f}")
                
                for d_id, d_name, w in zip(drug_ids, drug_names, weights):
                    viz_rows.append({
                        'Combination': combo_name,
                        'DrugBank_ID': d_id,
                        'Drug_Name': d_name,
                        'Side_Effect_CUI': cui,
                        'Side_Effect_Name': se_label,
                        'Attention_Weight': w,
                        'Predicted_Probability': prob
                    })
                    
    viz_df = pd.DataFrame(viz_rows)
    viz_path = os.path.join(output_dir, 'attention_visualization.csv')
    viz_df.to_csv(viz_path, index=False)
    print(f"\nAttention diagnostics table saved to: {viz_path}")
    return viz_df

In [ ]:
# ------------------------------------------------------------------------------
# Step 5: Post-Training All-Side-Effects Screening Engine
# ------------------------------------------------------------------------------
def screen_all_side_effects(model, drug_ids, data, device, alert_threshold=0.30, top_k=20):
    """
    Evaluates any arbitrary drug combination against ALL 7,350 MedDRA side effects in parallel.
    Filters high-risk alerts using the 30% medical safety threshold and identifies culprit drugs.
    """
    model.eval()
    drug_to_index = data['drug_to_index']
    drug_names_dict = data['drug_names_dict']
    se_df = data['se_df']
    num_drugs = data['num_drugs']
    drug_features = data['drug_features']
    
    valid_query_drugs = [d for d in drug_ids if d in drug_to_index]
    if len(valid_query_drugs) == 0:
        print("Error: None of the provided DrugBank IDs were found in the vocabulary.")
        return None
        
    query_names = [drug_names_dict.get(d, d) for d in valid_query_drugs]
    print(f"\n=======================================================")
    print(f" FULL-SPECTRUM ADVERSE EVENT SCREENING ({len(valid_query_drugs)} Drugs)")
    print(f" Drugs Tested: {', '.join(query_names)}")
    print(f" Scanning {len(se_df)} unique MedDRA adverse events simultaneously...")
    print(f" Safety Alert Threshold: {alert_threshold*100:.0f}%")
    print(f"=======================================================")
    
    H_single = torch.zeros((num_drugs, 1), dtype=torch.float32).to(device)
    for d in valid_query_drugs:
        H_single[drug_to_index[d], 0] = 1.0
        
    feature_cols = [str(i) for i in range(768)]
    all_se_feats = torch.tensor(se_df[feature_cols].values.astype(np.float32), device=device)
    num_se = len(se_df)
    H_T_expanded = H_single.T.repeat(num_se, 1)
    
    with torch.no_grad():
        # 1. Forward drug hypergraph representation once
        X = model.drug_encoder(drug_features)
        row, col = torch.where(H_single.T)
        edges = torch.cat([col.view(1, -1), row.view(1, -1)], dim=0).to(device)
        degree_e = H_single.T.sum(dim=1, keepdim=True).clamp(min=1)
        
        attr1 = (H_single.T @ X) / degree_e
        X1 = F.relu(model.conv1(X, edges, hyperedge_attr=attr1))
        attr2 = (H_single.T @ X1) / degree_e
        X2 = F.relu(model.conv2(X1, edges, hyperedge_attr=attr2))
        
        # 2. Forward all side effects
        h_se_all = model.se_encoder(all_se_feats)
        h_e_mean = (H_single.T @ X2) / degree_e
        h_e_mean_rep = h_e_mean.repeat(num_se, 1)
        
        # 3. Dynamic Attention Pooling
        q = model.W_q(torch.cat([h_e_mean_rep, h_se_all], dim=1))
        k = model.W_k(X2)
        v = model.W_v(X2)
        
        scores = torch.matmul(q, k.T) / math.sqrt(model.d)
        scores = scores.masked_fill(H_T_expanded == 0, -1e9)
        alpha = F.softmax(scores, dim=1)
        
        h_combo_all = torch.matmul(alpha, v)
        z = torch.cat([h_combo_all, h_se_all], dim=1)
        probs = torch.sigmoid(model.decoder(z).squeeze(-1))
        
        # 4. Rank highest risks
        top_probs, top_indices = torch.topk(probs, k=min(top_k, num_se))
        
        results = []
        for rank, (p_val, idx) in enumerate(zip(top_probs.tolist(), top_indices.tolist()), start=1):
            cui = se_df.iloc[idx]['umls_cui_from_meddra']
            se_name = se_df.iloc[idx]['side_effect_name']
            attns = {query_names[i]: round(alpha[idx, drug_to_index[valid_query_drugs[i]]].item(), 4) for i in range(len(valid_query_drugs))}
            culprit = max(attns, key=attns.get)
            
            # Assign Clinical Risk Tier
            if p_val >= 0.70:
                tier = "🔴 High Risk (>=70%)"
            elif p_val >= alert_threshold:
                tier = "🟡 Moderate Risk (30-70%)"
            else:
                tier = "🟢 Low Risk (<30%)"
                
            results.append({
                'Rank': rank,
                'Side Effect CUI': cui,
                'Side Effect Name': se_name,
                'Probability': f"{p_val * 100:.2f}%",
                'Raw Prob': p_val,
                'Risk Tier': tier,
                'Culprit Drug': culprit,
                'Drug Attention Weights': attns
            })
            
    res_df = pd.DataFrame(results)
    return res_df

In [ ]:
# ------------------------------------------------------------------------------
# Step 6: Training Pipeline (Exp C: HyperAttDDI)
# ------------------------------------------------------------------------------
set_random_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing on Device: {device}")

base_path = find_dataset_path()
data = load_data(base_path, device)
drug_features = data['drug_features']
train_inc = data['train_inc']
train_se = data['train_se']
y_train = data['train_lab']

# Training Configuration
epochs = 100
batch_size = 64
eval_batch_size = 512
lr = 0.0005
weight_decay = 0.001

model = HyperAttDDI_SE(
    in_dim=768,
    emb_dim=256,
    conv_dim=64,
    heads=4,
    d=64,
    p=0.1
).to(device)

model.apply(init_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
criterion = nn.BCEWithLogitsLoss()

print(f"\nModel initialized with {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable parameters.")
print("Starting Model Training...\n")

max_valid_f1 = 0.0
best_epoch = -1
best_weights = None

num_samples = train_inc.shape[1]
num_batches = num_samples // batch_size

for epoch in range(epochs):
    model.train()
    epoch_loss = 0.0
    
    indices = torch.randperm(num_samples)
    shuffled_train_inc = train_inc[:, indices]
    shuffled_train_se = train_se[indices]
    shuffled_y_train = y_train[indices]
    
    pbar = tqdm(range(num_batches), desc=f"Epoch {epoch+1:03d}/{epochs:03d}", leave=False)
    for b in pbar:
        optimizer.zero_grad()
        batch_inc = shuffled_train_inc[:, b * batch_size:(b + 1) * batch_size].to(device)
        batch_se = shuffled_train_se[b * batch_size:(b + 1) * batch_size].to(device)
        batch_y = shuffled_y_train[b * batch_size:(b + 1) * batch_size].to(device)
        
        logits = model(drug_features, batch_inc, batch_se)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
        
    avg_loss = epoch_loss / num_batches
    
    # Validate every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == epochs - 1:
        val_res = evaluate_dataset(
            model, drug_features, data['val_inc'], data['val_se'], data['val_lab'], device, batch_size=eval_batch_size
        )
        val_f1 = val_res['f1']
        val_auc = val_res['auc']
        
        print(f"Epoch {epoch+1:03d} | Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val AUC: {val_auc:.4f} | Val Precision: {val_res['precision']:.4f} | Val Recall: {val_res['recall']:.4f}")
        
        if val_f1 > max_valid_f1:
            max_valid_f1 = val_f1
            best_epoch = epoch + 1
            best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            torch.save(best_weights, 'best_hyperattddi_exp_c.pt')
            print(f"  --> Saved new best model checkpoint (Val F1: {val_f1:.4f})")

print(f"\nTraining Complete! Best Validation F1: {max_valid_f1:.4f} at Epoch {best_epoch}.")

In [ ]:
# ------------------------------------------------------------------------------
# Step 7: Dual-Threshold Test Evaluation (0.50 Standard vs 0.30 Medical Safety)
# ------------------------------------------------------------------------------
if best_weights is not None:
    model.load_state_dict(best_weights)
model.to(device)

# 1. Standard Benchmark Evaluation (Threshold = 0.50)
test_res_std = evaluate_dataset(
    model, drug_features, data['test_inc'], data['test_se'], data['test_lab'], device, batch_size=eval_batch_size, threshold=0.50
)

# 2. High-Sensitivity Medical Safety Evaluation (Threshold = 0.30)
test_res_safe = evaluate_dataset(
    model, drug_features, data['test_inc'], data['test_se'], data['test_lab'], device, batch_size=eval_batch_size, threshold=0.30
)

print("\n=======================================================")
print("         FINAL TEST PERFORMANCE (Experiment C)         ")
print("=======================================================")
print(f"[Standard Evaluation (Threshold = 0.50)]")
print(f"  ROC-AUC   : {test_res_std['auc']:.4f}")
print(f"  PR-AUC    : {test_res_std['prauc']:.4f}")
print(f"  Precision : {test_res_std['precision']:.4f}")
print(f"  Recall    : {test_res_std['recall']:.4f}")
print(f"  F1-Score  : {test_res_std['f1']:.4f}")
print("-------------------------------------------------------")
print(f"[Medical Safety Mode (Threshold = 0.30 - High Sensitivity)]")
print(f"  Precision : {test_res_safe['precision']:.4f}")
print(f"  Recall    : {test_res_safe['recall']:.4f} (Maximized Clinical Detection)")
print(f"  F1-Score  : {test_res_safe['f1']:.4f}")
print("=======================================================")

# Export Attention Shift Diagnostics
viz_df = perform_attention_visualization(model, drug_features, data, device, output_dir='.')

In [ ]:
# ------------------------------------------------------------------------------
# Step 8: Interactive Multi-Drug Adverse Event Screening Demo
# ------------------------------------------------------------------------------
# Example 1: Cardiovascular Prescription (Clopidogrel + Dabigatran + Aspirin)
cardio_prescription = ['DB00758', 'DB14726', 'DB00945']
screen_cardio = screen_all_side_effects(
    model, cardio_prescription, data, device, alert_threshold=0.30, top_k=10
)
print("\n--- Top Predicted Adverse Events (Cardiovascular Prescription) ---")
display(screen_cardio[['Rank', 'Side Effect Name', 'Probability', 'Risk Tier', 'Culprit Drug']])

# Example 2: CNS Neuro-Psychiatric Prescription (Escitalopram + Clobazam + Pregabalin + Tramadol)
cns_prescription = ['DB01175', 'DB00349', 'DB00230', 'DB00193']
screen_cns = screen_all_side_effects(
    model, cns_prescription, data, device, alert_threshold=0.30, top_k=10
)
print("\n--- Top Predicted Adverse Events (CNS Neuro-Psychiatric Prescription) ---")
display(screen_cns[['Rank', 'Side Effect Name', 'Probability', 'Risk Tier', 'Culprit Drug']])